# GBD 2023 Trend Analysis — All 12 NCD Level 2 Categories

**Disease Burden Among Aging Canadians — Trend Analysis (Extended)**

Extends the original 7-category trend analysis to all 12 GBD Level 2 categories within the Non-Communicable Diseases umbrella, per supervisor direction.

Produces:
- **Table 1** — Disease burden by category, 2023 (DALYs, deaths, growth, rank)
- **Table 2** — Year-by-year absolute DALY trends, 1995–2023
- **Table 3** — Age-standardized DALY rate trends, 1995–2023

**Scope:** Canada, both sexes, age 60+, years 1995/2000/2005/2010/2015/2019/2023.

**Data provenance:**
- Original 7 categories (Cardiovascular diseases, Neoplasms, Neurological disorders, Musculoskeletal disorders, Diabetes and kidney diseases, Chronic respiratory diseases, Mental disorders) — verified against the original GBD 2023 category-specific exports used in the published paper's Tables 1–3.
- 5 new categories (Digestive diseases, Substance use disorders, Skin and subcutaneous diseases, Sense organ diseases, Other non-communicable diseases) — loaded directly from `IHME-GBD_2023_DATA-13912a7c-1_new_level2_csv.xlsx`, the raw GBD Results Tool export pulled for this extension.

**Note on "Other non-communicable diseases":** this is GBD's catch-all Level 2 grouping (congenital anomalies, oral disorders, and other small causes not assigned elsewhere). Its trend should be reported but not over-interpreted as a single clinical entity.

**Note on missing death data:** GBD does not report mortality for two categories in this set — Mental disorders and Sense organ diseases — since both are primarily disability-driving conditions with negligible direct fatality. These are reported as N/A, consistent with the original paper's treatment of Mental disorders.

In [3]:
import pandas as pd
import numpy as np
import os

pd.set_option('display.width', 140)
pd.set_option('display.float_format', lambda x: f'{x:,.1f}')

## 1. Load data

Original 7 categories are embedded as verified constants (matching the published paper's Tables 1–3 exactly). The 5 new categories are loaded directly from the raw GBD export file if present; if the file isn't found, verified fallback constants are used instead so this notebook still runs end-to-end.

In [4]:
OBS_YEARS = [1995, 2000, 2005, 2010, 2015, 2019, 2023]

# ── Original 7 categories — verified against the published paper (Tables 1–3) ──
daly_original = {
    'Neoplasms':                    [973032, 1023542, 1080966, 1163133, 1294164, 1394261, 1557828],
    'Cardiovascular diseases':      [1196259, 1144752, 1078819, 1051908, 1126587, 1200679, 1333532],
    'Neurological disorders':       [246959, 286371, 331147, 387131, 456361, 516731, 582179],
    'Musculoskeletal disorders':    [259358, 282590, 313597, 364370, 433771, 500108, 552267],
    'Chronic respiratory diseases': [219540, 239279, 256933, 282544, 334880, 374449, 415842],
    'Diabetes and kidney diseases': [172889, 207633, 243272, 257874, 282879, 322291, 374726],
    'Mental disorders':             [74447, 79391, 90221, 108909, 129190, 148278, 194075],
}

rate_original = {
    'Cardiovascular diseases':      [25394.7, 22312.9, 18586.5, 15338.9, 13962.4, 13109.0, 13055.7],
    'Neoplasms':                    [20655.9, 19950.3, 18623.5, 16960.8, 16039.2, 15222.5, 15251.7],
    'Neurological disorders':       [5242.5,  5581.8,  5705.2,  5645.1,  5655.9,  5641.7,  5699.7],
    'Musculoskeletal disorders':    [5505.8,  5508.1,  5402.8,  5313.2,  5375.9,  5460.2,  5406.9],
    'Chronic respiratory diseases': [4660.5,  4663.9,  4426.6,  4120.1,  4150.3,  4088.2,  4071.2],
    'Diabetes and kidney diseases': [3670.2,  4047.1,  4191.2,  3760.3,  3505.9,  3518.8,  3668.7],
    'Mental disorders':             [1580.4,  1547.4,  1554.4,  1588.1,  1601.1,  1618.9,  1900.1],
}

deaths_2023_original = {
    'Neoplasms': 88335, 'Cardiovascular diseases': 78305,
    'Neurological disorders': 28739, 'Chronic respiratory diseases': 19734,
    'Diabetes and kidney diseases': 15384, 'Musculoskeletal disorders': 1142,
    'Mental disorders': None,  # GBD reports no death data for this primarily-disability category
}

In [8]:
# ── 5 new categories — loaded from raw GBD export, with verified fallback ──
NEW_FIVE = ['Digestive diseases', 'Substance use disorders', 'Skin and subcutaneous diseases',
            'Sense organ diseases', 'Other non-communicable diseases']

GBD_NEW_FILE = "C:\\Users\\amala\\Downloads\\IHME-GBD_2023_DATA-13912a7c-1_new_level2.csv.xlsx"

def load_new_categories(filepath):
    """Load DALYs (Number + Rate) and Deaths for the 5 new NCD categories from the raw
    GBD Results Tool export. Returns (daly_dict, rate_dict, deaths_dict)."""
    df = pd.read_excel(filepath)
    daly, rate, deaths = {}, {}, {}
    for cause in NEW_FIVE:
        daly_sub = df[(df['cause_name'] == cause) & (df['metric_name'] == 'Number') &
                      (df['measure_name'] == 'DALYs (Disability-Adjusted Life Years)') &
                      (df['age_name'] == '60+ years')].sort_values('year')
        daly[cause] = [round(v) for v in daly_sub['val'].tolist()]

        rate_sub = df[(df['cause_name'] == cause) & (df['metric_name'] == 'Rate') &
                      (df['measure_name'] == 'DALYs (Disability-Adjusted Life Years)') &
                      (df['age_name'] == '60+ years')].sort_values('year')
        rate[cause] = [round(v, 1) for v in rate_sub['val'].tolist()]

        death_sub = df[(df['cause_name'] == cause) & (df['metric_name'] == 'Number') &
                       (df['measure_name'] == 'Deaths') &
                       (df['age_name'] == '60+ years')].sort_values('year')
        deaths[cause] = round(death_sub['val'].iloc[-1]) if len(death_sub) > 0 else None
    return daly, rate, deaths


# Verified fallback (identical to what load_new_categories() returns from the raw file —
# embedded here so this notebook still runs if the raw export isn't in the working directory)
daly_new_fallback = {
    'Digestive diseases':              [129061, 135067, 147968, 165586, 196872, 226955, 274238],
    'Substance use disorders':         [19119, 19818, 22299, 28417, 38850, 48394, 61873],
    'Skin and subcutaneous diseases':  [18263, 21343, 24451, 29243, 35866, 41111, 48238],
    'Sense organ diseases':            [126257, 139698, 161003, 174447, 202071, 242485, 274025],
    'Other non-communicable diseases': [88070, 100156, 124373, 145828, 159825, 180552, 207574],
}
rate_new_fallback = {
    'Digestive diseases':              [2739.8, 2632.6, 2549.3, 2414.6, 2439.9, 2477.9, 2684.9],
    'Substance use disorders':         [405.9, 386.3, 384.2, 414.4, 481.5, 528.4, 605.8],
    'Skin and subcutaneous diseases':  [387.7, 416.0, 421.3, 426.4, 444.5, 448.9, 472.3],
    'Sense organ diseases':            [2680.2, 2722.9, 2773.9, 2543.8, 2504.4, 2647.4, 2682.8],
    'Other non-communicable diseases': [1869.6, 1952.2, 2142.8, 2126.5, 1980.8, 1971.3, 2032.2],
}
deaths_new_fallback = {
    'Digestive diseases': 13783, 'Substance use disorders': 1684,
    'Skin and subcutaneous diseases': 998, 'Sense organ diseases': None,  # no death data in GBD for this category
    'Other non-communicable diseases': 5904,
}

if os.path.exists(GBD_NEW_FILE):
    daly_new, rate_new, deaths_new = load_new_categories(GBD_NEW_FILE)
    print(f"✓ Loaded 5 new categories from raw file: {GBD_NEW_FILE}")
else:
    daly_new, rate_new, deaths_new = daly_new_fallback, rate_new_fallback, deaths_new_fallback
    print(f"⚠ Raw file '{GBD_NEW_FILE}' not found in working directory — using verified fallback constants instead.")

✓ Loaded 5 new categories from raw file: C:\Users\amala\Downloads\IHME-GBD_2023_DATA-13912a7c-1_new_level2.csv.xlsx


In [9]:
# ── Merge into the full 12-category dataset ──
daly_data = {**daly_original, **daly_new}
rate_data = {**rate_original, **rate_new}
deaths_2023 = {**deaths_2023_original, **deaths_new}

assert len(daly_data) == 12, f"Expected 12 categories, got {len(daly_data)}"
assert len(rate_data) == 12, f"Expected 12 categories, got {len(rate_data)}"
print(f"✓ {len(daly_data)} disease categories loaded:")
for c in daly_data:
    print(f"   - {c}")

✓ 12 disease categories loaded:
   - Neoplasms
   - Cardiovascular diseases
   - Neurological disorders
   - Musculoskeletal disorders
   - Chronic respiratory diseases
   - Diabetes and kidney diseases
   - Mental disorders
   - Digestive diseases
   - Substance use disorders
   - Skin and subcutaneous diseases
   - Sense organ diseases
   - Other non-communicable diseases


## 2. Table 1 — Disease burden by category, 2023

DALYs, deaths, growth 1995–2023, and DALY rank, for all 12 categories.

In [10]:
rows = []
for cause, vals in daly_data.items():
    growth = (vals[-1] - vals[0]) / vals[0] * 100
    rows.append({
        'Disease Category': cause,
        'DALYs (2023)': vals[-1],
        'Deaths (2023)': deaths_2023.get(cause) if deaths_2023.get(cause) is not None else 'N/A',
        'DALY Growth 1995–2023': f"+{growth:.1f}%",
    })

table1 = pd.DataFrame(rows).sort_values('DALYs (2023)', ascending=False).reset_index(drop=True)
table1.insert(0, 'DALY Rank', range(1, len(table1) + 1))
table1 = table1.set_index('DALY Rank')
table1

,Disease Category,DALYs (2023),Deaths (2023),DALY Growth 1995–2023
DALY Rank,,,,
1,Neoplasms,1557828,88335,+60.1%
2,Cardiovascular diseases,1333532,78305,+11.5%
3,Neurological disorders,582179,28739,+135.7%
4,Musculoskeletal disorders,552267,1142,+112.9%
5,Chronic respiratory diseases,415842,19734,+89.4%
6,Diabetes and kidney diseases,374726,15384,+116.7%
7,Digestive diseases,274238,13783,+112.5%
8,Sense organ diseases,274025,N/A,+117.0%
9,Other non-communicable diseases,207574,5904,+135.7%


In [11]:
total_2023 = sum(v[-1] for v in daly_data.values())
total_1995 = sum(v[0] for v in daly_data.values())
total_growth = (total_2023 - total_1995) / total_1995 * 100
deaths_2023_sum = sum(d for d in deaths_2023.values() if d is not None)

print(f"Total DALYs across 12 categories, 2023: {total_2023:,}")
print(f"Total DALYs across 12 categories, 1995: {total_1995:,}")
print(f"Growth 1995–2023: +{total_growth:.1f}%")
print(f"Total deaths across categories with death data, 2023: {deaths_2023_sum:,}")
print(f"(Categories with no death data: {[c for c,d in deaths_2023.items() if d is None]})")

Total DALYs across 12 categories, 2023: 5,876,397
Total DALYs across 12 categories, 1995: 3,523,254
Growth 1995–2023: +66.8%
Total deaths across categories with death data, 2023: 254,008
(Categories with no death data: ['Mental disorders', 'Sense organ diseases'])


## 3. Table 2 — Year-by-year absolute DALY trends, 1995–2023

Sorted by % change descending, matching the original paper's Table 2 convention.

In [12]:
table2 = pd.DataFrame(daly_data, index=OBS_YEARS).T
table2.index.name = 'Cause'
table2['% Change'] = table2.apply(lambda row: (row.iloc[-1] - row.iloc[0]) / row.iloc[0] * 100, axis=1)
table2 = table2.sort_values('% Change', ascending=False)
table2_display = table2.copy()
table2_display['% Change'] = table2_display['% Change'].apply(lambda x: f"+{x:.1f}%")
table2_display

,1995,2000,2005,2010,2015,2019,2023,% Change
Cause,,,,,,,,
Substance use disorders,19119,19818,22299,28417,38850,48394,61873,+223.6%
Skin and subcutaneous diseases,18263,21343,24451,29243,35866,41111,48238,+164.1%
Mental disorders,74447,79391,90221,108909,129190,148278,194075,+160.7%
Neurological disorders,246959,286371,331147,387131,456361,516731,582179,+135.7%
Other non-communicable diseases,88070,100156,124373,145828,159825,180552,207574,+135.7%
Sense organ diseases,126257,139698,161003,174447,202071,242485,274025,+117.0%
Diabetes and kidney diseases,172889,207633,243272,257874,282879,322291,374726,+116.7%
Musculoskeletal disorders,259358,282590,313597,364370,433771,500108,552267,+112.9%
Digestive diseases,129061,135067,147968,165586,196872,226955,274238,+112.5%


## 4. Table 3 — Age-standardized DALY rate trends, 1995–2023

Controls for population growth. Interpretation thresholds match the original paper's convention: rate change > +5% = "Rising"; between 0% and +5% = "Stable"; −0% to −10% = "Slightly declining"; below −10% = "Declining (effective)".

In [13]:
def interpret(change_pct):
    if change_pct > 5:
        return 'Rising (genuine)'
    elif change_pct > 0:
        return 'Stable'
    elif change_pct > -10:
        return 'Slightly declining'
    else:
        return 'Declining (effective)'

table3 = pd.DataFrame(rate_data, index=OBS_YEARS).T
table3.index.name = 'Cause'
table3['Rate Change'] = table3.apply(lambda row: (row.iloc[-1] - row.iloc[0]) / row.iloc[0] * 100, axis=1)
table3['Interpretation'] = table3['Rate Change'].apply(interpret)
table3 = table3.sort_values('Rate Change', ascending=False)
table3_display = table3.copy()
table3_display['Rate Change'] = table3_display['Rate Change'].apply(lambda x: f"{x:+.1f}%")
table3_display

,1995,2000,2005,2010,2015,2019,2023,Rate Change,Interpretation
Cause,,,,,,,,,
Substance use disorders,405.9,386.3,384.2,414.4,481.5,528.4,605.8,+49.2%,Rising (genuine)
Skin and subcutaneous diseases,387.7,416.0,421.3,426.4,444.5,448.9,472.3,+21.8%,Rising (genuine)
Mental disorders,"1,580.4","1,547.4","1,554.4","1,588.1","1,601.1","1,618.9","1,900.1",+20.2%,Rising (genuine)
Neurological disorders,"5,242.5","5,581.8","5,705.2","5,645.1","5,655.9","5,641.7","5,699.7",+8.7%,Rising (genuine)
Other non-communicable diseases,"1,869.6","1,952.2","2,142.8","2,126.5","1,980.8","1,971.3","2,032.2",+8.7%,Rising (genuine)
Sense organ diseases,"2,680.2","2,722.9","2,773.9","2,543.8","2,504.4","2,647.4","2,682.8",+0.1%,Stable
Diabetes and kidney diseases,"3,670.2","4,047.1","4,191.2","3,760.3","3,505.9","3,518.8","3,668.7",-0.0%,Slightly declining
Musculoskeletal disorders,"5,505.8","5,508.1","5,402.8","5,313.2","5,375.9","5,460.2","5,406.9",-1.8%,Slightly declining
Digestive diseases,"2,739.8","2,632.6","2,549.3","2,414.6","2,439.9","2,477.9","2,684.9",-2.0%,Slightly declining


## 5. Headline finding check

The original 7-category paper's central claim was that **Mental disorders** showed the steepest combination of absolute DALY growth (+160.7%) and age-standardized rate increase (+20.2%) of any category — framed as the most urgently under-addressed burden category. With 5 new categories added, that claim needs re-checking before any paper text is rewritten.

In [14]:
print("Ranked by absolute DALY growth, 1995–2023:")
print(table2['% Change'].sort_values(ascending=False).apply(lambda x: f"+{x:.1f}%").to_string())
print()
print("Ranked by age-standardized rate change, 1995–2023:")
print(table3['Rate Change'].sort_values(ascending=False).apply(lambda x: f"{x:+.1f}%").to_string())

Ranked by absolute DALY growth, 1995–2023:
Cause
Substance use disorders            +223.6%
Skin and subcutaneous diseases     +164.1%
Mental disorders                   +160.7%
Neurological disorders             +135.7%
Other non-communicable diseases    +135.7%
Sense organ diseases               +117.0%
Diabetes and kidney diseases       +116.7%
Musculoskeletal disorders          +112.9%
Digestive diseases                 +112.5%
Chronic respiratory diseases        +89.4%
Neoplasms                           +60.1%
Cardiovascular diseases             +11.5%

Ranked by age-standardized rate change, 1995–2023:
Cause
Substance use disorders            +49.2%
Skin and subcutaneous diseases     +21.8%
Mental disorders                   +20.2%
Neurological disorders              +8.7%
Other non-communicable diseases     +8.7%
Sense organ diseases                +0.1%
Diabetes and kidney diseases        -0.0%
Musculoskeletal disorders           -1.8%
Digestive diseases                  -2.0%

**Result:** Substance use disorders shows the highest growth of any of the 12 categories by **both** measures — absolute DALY growth (+223.6%, well above Mental disorders' +160.7%, previously the highest of the original 7) **and** age-standardized rate change (+49.2%, more than double Mental disorders' +20.2%). This is a substantive change to the paper's central finding, not a cosmetic one: under the original 7-category scope, Mental disorders was correctly identified as the most rapidly worsening category by both measures. With the 5 additional NCD categories included, that is no longer the case — Substance use disorders now holds that position outright, with Mental disorders second by both measures.

Skin and subcutaneous diseases is also notable: third-highest by both absolute growth (+164.1%) and rate change (+21.8%), surpassing Mental disorders on the absolute measure and landing just above it on the rate measure.

This needs to be reflected in the paper's Abstract, Results, Discussion, and Conclusion before any other section is updated — every place the manuscript currently names Mental disorders as "the steepest growth" or "most rapidly worsening" category needs to be revisited against this 12-category result.

## 6. Export

Save the three tables for use in the LaTeX manuscript and dashboard.

In [15]:
table1.to_csv('table1_burden_2023_12cat.csv')
table2_display.to_csv('table2_daly_trends_12cat.csv')
table3_display.to_csv('table3_rate_trends_12cat.csv')
print("Exported: table1_burden_2023_12cat.csv, table2_daly_trends_12cat.csv, table3_rate_trends_12cat.csv")

Exported: table1_burden_2023_12cat.csv, table2_daly_trends_12cat.csv, table3_rate_trends_12cat.csv
